<a href="https://colab.research.google.com/github/sunonmountain/High-Intent-Prospecting-Stack/blob/main/Harvester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import requests
import pandas as pd
from google.colab import userdata
SERPER_KEY = userdata.get("SERPER_API_KEY")

# --- CONFIGURATION ---
TEST_MODE = True

# 1. DOMAIN BLACKLIST: Hard blocks for regulators, news, and events
DOMAIN_BLACKLIST = [
    "bankofengland.co.uk", "fca.org.uk", "reuters.com", "bloomberg.com",
    "theguardian.com", "bbc.co.uk", "ft.com", "finextra.com", "ukfintechweek.co.uk",
    "crunchbase.com", "wikipedia.org", "clutch.co"
]

# 2. REFINED JUNK: Blocks general noise (jobs, students) but ALLOWS services/partners
JUNK_KEYWORDS = [
    "simplyhired", "indeed", "glassdoor", "job", "career", "hiring",
    "ac.uk", "university", "college", "prospectus", "gov.uk",
    "article", "journal", "forum", "reddit"
]

def is_valid_b2b_prospect(url, title):
    combined = (url + " " + title).lower()

    # Check Blacklist
    if any(blacklisted in url.lower() for blacklisted in DOMAIN_BLACKLIST):
        return False

    # Check Junk Keywords
    if any(junk in combined for junk in JUNK_KEYWORDS):
        return False

    return True

def run_harvester():
    cities = ["London"] if TEST_MODE else ["London", "Manchester", "Birmingham"]
    max_res = 10 if TEST_MODE else 100

    clean_leads = []
    print(f"🚀 Harvester Live: Sourcing B2B Services & Fintechs...")

    for city in cities:
        # We query specifically for Fintech services to trigger the right results
        query = f'site:.co.uk "Fintech" {city} "Partners" OR "Consulting"'

        url = "https://google.serper.dev/search"
        payload = {"q": query, "gl": "gb", "num": max_res}
        headers = {'X-API-KEY': SERPER_KEY, 'Content-Type': 'application/json'}

        try:
            response = requests.post(url, json=payload, headers=headers)
            results = response.json().get('organic', [])

            for result in results:
                link, title = result.get('link'), result.get('title')

                if is_valid_b2b_prospect(link, title):
                    domain = link.split('//')[-1].split('/')[0].replace('www.', '')
                    clean_leads.append({
                        "company_name": title.split('|')[0].strip(),
                        "domain": domain,
                        "city": city
                    })
                    print(f"   ✅ B2B Prospect Found: {domain}")
                else:
                    print(f"   🗑️ Filtered (Regulator/Junk): {link[:30]}")

        except Exception as e: print(f"❌ Error: {e}")

    df = pd.DataFrame(clean_leads)
    if not df.empty:
        df.drop_duplicates(subset=['domain'], inplace=True)
        df.to_csv("harvested_B2B_final.csv", index=False)
        print(f"🏁 Finished. {len(df)} High-Value B2B prospects saved.")

if __name__ == "__main__":
    run_harvester()

🚀 Harvester Live: Sourcing B2B Services & Fintechs...
   ✅ B2B Prospect Found: ec1partners.co.uk
   ✅ B2B Prospect Found: penser.co.uk
   ✅ B2B Prospect Found: fintechaurapartners.co.uk
   ✅ B2B Prospect Found: fintechaurapartners.co.uk
   ✅ B2B Prospect Found: fintechaurapartners.co.uk
   ✅ B2B Prospect Found: mooreks.co.uk
   🗑️ Filtered (Regulator/Junk): https://www.bankofengland.co.u
   ✅ B2B Prospect Found: whitecapconsulting.co.uk
   ✅ B2B Prospect Found: mooreks.co.uk
   ✅ B2B Prospect Found: futureunite.co.uk
🏁 Finished. 6 High-Value B2B prospects saved.
